# 00 · Setup

실행 방식 3: 설치 가능 패키지. 학습 로직은 [`custom_packages/src/recommender_pkg/`](./custom_packages/src/recommender_pkg/)의 `model.py` / `torch_distributor_trainer.py` / `lightning_trainer.py`에 있고, 노트북은 wheel 빌드·설치 + driver(파라미터·실행)만 담당합니다.

이 노트북은 후속 노트북(`01-data_prep`, `02..10`)에서 `%run ./00-setup`으로 불러 변수를 공유합니다.

정의 항목
- wheel 빌드 (`uv build`) + 설치 (`%pip install dist/*.whl`)
- 추가 패키지 (`lightning`, `nvidia-ml-py`)
- UC catalog / schema / volume 경로
- 단일 `CONFIG` dict (모델 하이퍼파라미터, batch, epoch)
- MovieLens 25M 다운로드 URL과 데이터 경로
- MLflow experiment 경로
- TorchDistributor child에 넘길 `db_host` / `db_token`

기준 모델·데이터셋: [`00-foundations/recommender-baseline.md`](../00-foundations/recommender-baseline.md)
클러스터: [`00-foundations/cluster-recipes.md`](../00-foundations/cluster-recipes.md)

## 빌드 도구 + 런타임 추가 패키지 설치

`uv`는 wheel 빌드에 사용하고, `lightning`·`nvidia-ml-py`는 02-script-based와 동일한 정책으로 런타임에 필요합니다.

| 패키지 | 용도 |
|--------|------|
| `uv` | `uv build`로 `recommender_pkg` wheel 생성 |
| `lightning==2.5.1` | Lightning trainer 노트북 (`05~07`) |
| `nvidia-ml-py` | MLflow `log_system_metrics=True`의 GPU 메트릭 수집 |

DBR 17.3 LTS ML에 사전 설치된 핵심 버전은 02-script-based/00-setup과 동일 (torch 2.7.0, mlflow-skinny 3.0.1, accelerate 1.5.2, CUDA 12.6, NCCL 2.26.2).

In [ ]:
%pip install --quiet uv "lightning==2.5.1" "nvidia-ml-py"
%restart_python

## `recommender_pkg` wheel 빌드

`custom_packages/` 디렉토리에서 `uv build`를 호출해 `dist/recommender_pkg-0.1.0-py3-none-any.whl`을 생성합니다. 빌드는 idempotent — 이미 있으면 덮어씁니다.

In [ ]:
import os
import subprocess

NOTEBOOK_PATH = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
NOTEBOOK_DIR = "/Workspace" + os.path.dirname(NOTEBOOK_PATH)
PKG_DIR = os.path.join(NOTEBOOK_DIR, "custom_packages")
WHEEL_PATH = os.path.join(PKG_DIR, "dist", "recommender_pkg-0.1.0-py3-none-any.whl")

print(f"PKG_DIR={PKG_DIR}")
subprocess.run(["uv", "build"], cwd=PKG_DIR, check=True)
assert os.path.exists(WHEEL_PATH), WHEEL_PATH
print(f"built: {WHEEL_PATH} ({os.path.getsize(WHEEL_PATH)/1024:.1f} KB)")

## wheel 설치

빌드된 wheel을 노트북 세션에 설치합니다. 이후 `from recommender_pkg.model import TwoTowerMLP` 등으로 import 가능합니다.

In [ ]:
%pip install --quiet $WHEEL_PATH
%restart_python

## UC 경로

자기 워크스페이스에 맞춰 `CATALOG`, `SCHEMA`, `VOLUME`을 수정합니다. 02-script-based와 같은 catalog/schema/volume을 써도 되고, 분리해도 됩니다 (`distributed_cookbook_pkg` 등).

In [ ]:
CATALOG = "main"
SCHEMA = "distributed_cookbook"
VOLUME = "dataset"

VOLUME_ROOT = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
DATA_DIR = f"{VOLUME_ROOT}/ml-25m"      # train/, val/, _meta.json 위치
CKPT_DIR = f"{VOLUME_ROOT}/checkpoints"

ML25M_URL = "https://files.grouplens.org/datasets/movielens/ml-25m.zip"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

import os
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

print(f"VOLUME_ROOT={VOLUME_ROOT}")
print(f"DATA_DIR={DATA_DIR}")
print(f"CKPT_DIR={CKPT_DIR}")

## MLflow experiment

워크스페이스 사용자 폴더 아래에 experiment를 만듭니다. 모든 토폴로지(1×1 / 1×N / M×N)는 같은 experiment에 run으로 누적됩니다. 02-script-based와 구분되도록 `recommender-custom-package`를 씁니다.

In [ ]:
import mlflow

USERNAME = (
    spark.sql("SELECT current_user() AS user").first()["user"]
)
EXPERIMENT_PATH = f"/Users/{USERNAME}/recommender-custom-package"

experiment = mlflow.get_experiment_by_name(EXPERIMENT_PATH)
if experiment is None:
    EXPERIMENT_ID = mlflow.create_experiment(EXPERIMENT_PATH)
else:
    EXPERIMENT_ID = experiment.experiment_id
mlflow.set_experiment(experiment_id=EXPERIMENT_ID)
print(f"EXPERIMENT_PATH={EXPERIMENT_PATH} (id={EXPERIMENT_ID})")

## 단일 CONFIG

02-script-based/00-setup과 동일한 단일 CONFIG. ML-25M 단일 데이터셋에 맞춘 단일 하이퍼파라미터. 1×1 / 1×N / M×N 토폴로지는 모델·데이터를 **공유**하고 launcher 설정과 world_size만 다릅니다.

| 키 | 값 | 비고 |
|----|-----|-----|
| `emb_dim` | 64 | 두 tower 공통 |
| `tower_hidden` | (256, 128) | |
| `batch_size_per_gpu` | 4096 | DDP에서 per-rank, global은 ×world_size |
| `num_epochs` | 10 | 상한 |
| `max_steps_per_epoch` | 200 | epoch당 step 캡 |
| `val_ratio` | 0.1 | data_prep에서 적용 |
| `n_users` | metadata에서 로드 | ML-25M 기준 ≈ 162,541 |
| `n_items` | metadata에서 로드 | ML-25M 기준 ≈ 59,047 |

In [ ]:
import json

CONFIG = {
    "emb_dim": 64,
    "tower_hidden": (256, 128),
    "batch_size_per_gpu": 4096,
    "num_epochs": 10,
    "max_steps_per_epoch": 200,
    "val_ratio": 0.1,
    "n_users": None,
    "n_items": None,
}

PATIENCE = 3
MIN_DELTA = 1e-4

meta_path = os.path.join(DATA_DIR, "_meta.json")
if os.path.exists(meta_path):
    with open(meta_path) as f:
        meta = json.load(f)
    CONFIG["n_users"] = meta["n_users"]
    CONFIG["n_items"] = meta["n_items"]
    print(f"loaded metadata: n_users={meta['n_users']:,} n_items={meta['n_items']:,} train_rows={meta.get('n_train_rows', 'n/a'):,} val_rows={meta.get('n_val_rows', 'n/a'):,}")
else:
    print(f"[warn] {meta_path} not found. Run 01-data_prep first before training.")

for k, v in CONFIG.items():
    print(f"  CONFIG[{k!r}] = {v}")
print(f"early stopping: patience={PATIENCE} min_delta={MIN_DELTA}")

## TorchDistributor child용 자격증명

TorchDistributor가 띄운 worker는 driver의 Databricks 자격증명을 자동 상속하지 않습니다. driver에서 host/token을 한 번 읽어두고, 학습 함수의 인자로 명시 전달합니다 ([common-pitfalls.md §2-1](../00-foundations/common-pitfalls.md)).

In [ ]:
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
DB_HOST = ctx.extraContext().apply("api_url")
DB_TOKEN = ctx.apiToken().get()
print(f"DB_HOST={DB_HOST}")